In [1]:
!python -c "import torch; print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0)}')"

CUDA: True, GPU: NVIDIA A40


# Stage 1: Train EfficientNet-B0 on Clean Audio Clips
Train a multi-label classifier on 35,549 audio clips (206 species with training data, 234 total target species).

In [2]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
import librosa
import soundfile as sf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── Config ──
CFG = {
    'seed': 42,
    'base_dir': '/data/scratch/scienceteam/jupyter-mir/bird/Data/birdclef-2026',
    'output_dir': '/data/scratch/scienceteam/jupyter-mir/bird/models/stage1',
    'sr': 32000,
    'duration': 5,          # seconds per clip
    'n_fft': 1024,
    'hop_length': 512,
    'n_mels': 128,
    'fmin': 20,
    'fmax': 16000,
    'img_width': 313,       # time frames for 5s at hop=512: (5*32000)//512 + 1 = 313
    'model_name': 'efficientnet_b0',
    'num_classes': 234,
    'batch_size': 32,
    'epochs': 20,
    'patience': 3, 
    'lr': 1e-3,
    'weight_decay': 1e-2,
    'mixup_alpha': 0.15,
    'n_folds': 5,
    'num_workers': 4,
}

# ── Seed everything ──
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['seed'])
os.makedirs(CFG['output_dir'], exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE} ({torch.cuda.get_device_name(0)})")

Device: cuda (NVIDIA A40)


In [3]:
train = pd.read_csv(os.path.join(CFG['base_dir'], 'train.csv'))
taxonomy = pd.read_csv(os.path.join(CFG['base_dir'], 'taxonomy.csv'))

# Build label-to-index mapping from taxonomy (all 234 species)
LABELS = sorted(taxonomy['primary_label'].tolist())
LABEL2IDX = {label: idx for idx, label in enumerate(LABELS)}
IDX2LABEL = {idx: label for label, idx in LABEL2IDX.items()}

# Add full file paths
train['filepath'] = train['filename'].apply(lambda x: os.path.join(CFG['base_dir'], 'train_audio', x))

# Verify files exist (quick check on a sample)
missing = train[~train['filepath'].apply(os.path.exists)]
print(f"Total training samples: {len(train):,}")
print(f"Total target species: {len(LABELS)}")
print(f"Species with training data: {train['primary_label'].nunique()}")
print(f"Missing files: {len(missing)}")

Total training samples: 35,549
Total target species: 234
Species with training data: 206
Missing files: 0


In [4]:
class BirdDataset(Dataset):
    def __init__(self, df, label2idx, cfg, is_train=True):
        self.df = df.reset_index(drop=True)
        self.label2idx = label2idx
        self.cfg = cfg
        self.is_train = is_train
        self.target_len = cfg['sr'] * cfg['duration']  # 160,000 samples for 5s
    
    def __len__(self):
        return len(self.df)
    
    def load_audio(self, filepath):
        """Load audio and extract a 5-second window."""
        try:
            y, sr = librosa.load(filepath, sr=self.cfg['sr'])
        except Exception:
            y = np.zeros(self.target_len, dtype=np.float32)
            return y
        
        if len(y) == 0:
            return np.zeros(self.target_len, dtype=np.float32)
        
        # If shorter than 5s, tile to fill
        if len(y) < self.target_len:
            y = np.tile(y, (self.target_len // len(y)) + 1)
        
        # Extract window
        if self.is_train and len(y) > self.target_len:
            # Random window during training
            start = random.randint(0, len(y) - self.target_len)
            y = y[start:start + self.target_len]
        else:
            # Center window during validation
            center = len(y) // 2
            half = self.target_len // 2
            y = y[center - half:center - half + self.target_len]
        
        return y.astype(np.float32)
    
    def audio_to_melspec(self, y):
        """Convert waveform to log-mel spectrogram."""
        mel = librosa.feature.melspectrogram(
            y=y, sr=self.cfg['sr'],
            n_fft=self.cfg['n_fft'],
            hop_length=self.cfg['hop_length'],
            n_mels=self.cfg['n_mels'],
            fmin=self.cfg['fmin'],
            fmax=self.cfg['fmax'],
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)
        
        # Normalize to [0, 1]
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
        return mel_db
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load and process audio
        y = self.load_audio(row['filepath'])
        mel = self.audio_to_melspec(y)
        
        # Pad/trim to fixed width
        if mel.shape[1] < self.cfg['img_width']:
            mel = np.pad(mel, ((0, 0), (0, self.cfg['img_width'] - mel.shape[1])))
        else:
            mel = mel[:, :self.cfg['img_width']]
        
        # Stack to 3 channels (EfficientNet expects 3-channel input)
        mel = np.stack([mel, mel, mel], axis=0)
        mel = torch.tensor(mel, dtype=torch.float32)
        
        # Multi-label target
        target = torch.zeros(self.cfg['num_classes'], dtype=torch.float32)
        if row['primary_label'] in self.label2idx:
            target[self.label2idx[row['primary_label']]] = 1.0
        
        return mel, target

# Quick test
test_ds = BirdDataset(train.head(2), LABEL2IDX, CFG, is_train=True)
mel, target = test_ds[0]
print(f"Mel shape: {mel.shape}")       # [3, 128, 313]
print(f"Target shape: {target.shape}")  # [234]
print(f"Active labels: {target.sum().item():.0f}")

Mel shape: torch.Size([3, 128, 313])
Target shape: torch.Size([234])
Active labels: 1


In [5]:
class BirdModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg['model_name'], pretrained=True,
            in_chans=3, num_classes=0  # remove default head
        )
        self.head = nn.Linear(self.backbone.num_features, cfg['num_classes'])
    
    def forward(self, x):
        features = self.backbone(x)
        logits = self.head(features)
        return logits

# Quick test
model = BirdModel(CFG).to(DEVICE)
dummy = torch.randn(2, 3, 128, 313).to(DEVICE)
out = model(dummy)
print(f"Model output shape: {out.shape}")  # [2, 234]
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
del model, dummy, out
torch.cuda.empty_cache()

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Model output shape: torch.Size([2, 234])
Total parameters: 4,307,302


In [6]:
def mixup(mel, target, alpha=0.15):
    """Blend two samples together for regularization."""
    if alpha <= 0:
        return mel, target
    lam = np.random.beta(alpha, alpha)
    indices = torch.randperm(mel.size(0))
    mel = lam * mel + (1 - lam) * mel[indices]
    target = lam * target + (1 - lam) * target[indices]
    return mel, target

def spec_augment(mel, freq_mask=20, time_mask=40):
    """Randomly mask frequency and time bands."""
    _, n_mels, n_time = mel.shape
    
    # Frequency mask
    f = random.randint(0, freq_mask)
    f0 = random.randint(0, max(0, n_mels - f))
    mel[:, f0:f0+f, :] = 0
    
    # Time mask
    t = random.randint(0, time_mask)
    t0 = random.randint(0, max(0, n_time - t))
    mel[:, :, t0:t0+t] = 0
    
    return mel

print("Mixup and SpecAugment defined.")

Mixup and SpecAugment defined.


In [7]:
def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    losses = []
    criterion = nn.BCEWithLogitsLoss()
    
    pbar = tqdm(loader, desc="Train")
    for mel, target in pbar:
        mel, target = mel.to(device), target.to(device)
        
        # Mixup
        mel, target = mixup(mel, target, alpha=CFG['mixup_alpha'])
        
        # SpecAugment (apply per sample)
        for i in range(mel.size(0)):
            if random.random() < 0.5:
                mel[i] = spec_augment(mel[i])
        
        logits = model(mel)
        loss = criterion(logits, target)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        pbar.set_postfix(loss=f"{np.mean(losses[-50:]):.4f}")
    
    scheduler.step()
    return np.mean(losses)


def validate(model, loader, device):
    model.eval()
    all_preds = []
    all_targets = []
    losses = []
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for mel, target in tqdm(loader, desc="Valid"):
            mel, target = mel.to(device), target.to(device)
            logits = model(mel)
            loss = criterion(logits, target)
            losses.append(loss.item())
            
            preds = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(target.cpu().numpy())
    
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    
    # Macro ROC-AUC (skip classes with no positives)
    aucs = []
    for i in range(all_targets.shape[1]):
        if all_targets[:, i].sum() > 0:
            try:
                auc = roc_auc_score(all_targets[:, i], all_preds[:, i])
                aucs.append(auc)
            except ValueError:
                pass
    
    macro_auc = np.mean(aucs) if aucs else 0.0
    return np.mean(losses), macro_auc, len(aucs)

print("Training functions defined.")

Training functions defined.


In [8]:
# Stratified split by primary_label
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])

fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train['primary_label'])):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}")
    print(f"{'='*60}")
    
    train_df = train.iloc[train_idx]
    val_df = train.iloc[val_idx]
    
    print(f"Train: {len(train_df):,} | Val: {len(val_df):,}")
    
    train_ds = BirdDataset(train_df, LABEL2IDX, CFG, is_train=True)
    val_ds = BirdDataset(val_df, LABEL2IDX, CFG, is_train=False)
    
    train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                              num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'] * 2, shuffle=False,
                            num_workers=CFG['num_workers'], pin_memory=True)
    
    model = BirdModel(CFG).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'], eta_min=1e-6)
    
    best_auc = 0
    no_improve = 0
    
    for epoch in range(CFG['epochs']):
        print(f"\n--- Epoch {epoch+1}/{CFG['epochs']} ---")
        
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, DEVICE)
        val_loss, val_auc, n_scored = validate(model, val_loader, DEVICE)
        
        print(f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} (on {n_scored} species)")
        
        if val_auc > best_auc:
            best_auc = val_auc
            no_improve = 0
            save_path = os.path.join(CFG['output_dir'], f'stage1_fold{fold}.pth')
            torch.save(model.state_dict(), save_path)
            print(f"  -> Saved best model (AUC={best_auc:.4f})")
        else:
            no_improve += 1
            print(f"  -> No improvement ({no_improve}/{CFG['patience']})")
            if no_improve >= CFG['patience']:
                print(f"  -> Early stopping at epoch {epoch+1}")
                break
    
    fold_results.append({'fold': fold, 'best_auc': best_auc})
    print(f"\nFold {fold} best AUC: {best_auc:.4f}")
    
    # Free memory
    del model, optimizer, scheduler, train_loader, val_loader, train_ds, val_ds
    torch.cuda.empty_cache()

# Summary
results_df = pd.DataFrame(fold_results)
print(f"\n{'='*60}")
print(f"CROSS-VALIDATION RESULTS")
print(f"{'='*60}")
print(results_df.to_string(index=False))
print(f"\nMean AUC: {results_df['best_auc'].mean():.4f} +/- {results_df['best_auc'].std():.4f}")


FOLD 0
Train: 28,439 | Val: 7,110

--- Epoch 1/20 ---


Valid: 100%|██████████| 112/112 [01:07<00:00,  1.65it/s]


Train loss: 0.0274 | Val loss: 0.0216 | Val AUC: 0.8382 (on 198 species)
  -> Saved best model (AUC=0.8382)

--- Epoch 2/20 ---


Valid: 100%|██████████| 112/112 [01:07<00:00,  1.65it/s]


Train loss: 0.0199 | Val loss: 0.0166 | Val AUC: 0.9136 (on 198 species)
  -> Saved best model (AUC=0.9136)

--- Epoch 3/20 ---


...

Train loss: 0.0108 | Val loss: 0.0089 | Val AUC: 0.9676 (on 198 species)
  -> No improvement (1/3)

--- Epoch 10/20 ---


Valid: 100%|██████████| 112/112 [01:12<00:00,  1.55it/s]


Train loss: 0.0101 | Val loss: 0.0088 | Val AUC: 0.9663 (on 198 species)
  -> No improvement (2/3)

--- Epoch 11/20 ---


Valid: 100%|██████████| 112/112 [01:11<00:00,  1.56it/s]


Train loss: 0.0097 | Val loss: 0.0085 | Val AUC: 0.9702 (on 198 species)
  -> No improvement (3/3)
  -> Early stopping at epoch 11

Fold 4 best AUC: 0.9721

CROSS-VALIDATION RESULTS
 fold  best_auc
    0  0.970541
    1  0.960267
    2  0.962785
    3  0.974608
    4  0.972086

Mean AUC: 0.9681 +/- 0.0062


In [9]:
# Load best fold-0 model and check predictions on a few samples
model = BirdModel(CFG).to(DEVICE)
model.load_state_dict(torch.load(os.path.join(CFG['output_dir'], 'stage1_fold0.pth'), map_location=DEVICE))
model.eval()

val_df = train.iloc[list(skf.split(train, train['primary_label']))[0][1]]
samples = val_df.sample(5, random_state=42)

val_ds = BirdDataset(samples, LABEL2IDX, CFG, is_train=False)

print("Sample predictions (top 5 species per clip):\n")
for i in range(len(val_ds)):
    mel, target = val_ds[i]
    row = samples.iloc[i]
    
    with torch.no_grad():
        logits = model(mel.unsqueeze(0).to(DEVICE))
        probs = torch.sigmoid(logits).cpu().numpy()[0]
    
    top5_idx = probs.argsort()[-5:][::-1]
    true_label = row['primary_label']
    
    print(f"True: {true_label} ({row['common_name']})")
    for idx in top5_idx:
        marker = " <-- CORRECT" if IDX2LABEL[idx] == true_label else ""
        print(f"  {IDX2LABEL[idx]:>15s}: {probs[idx]:.4f}{marker}")
    print()

del model
torch.cuda.empty_cache()
print("Stage 1 complete!")

Sample predictions (top 5 species per clip):

True: masgna1 (Masked Gnatcatcher)
          masgna1: 0.9269 <-- CORRECT
          rufgna3: 0.0948
          ashgre1: 0.0342
          sobtyr1: 0.0189
          souant1: 0.0089

True: shtnig1 (Short-tailed Nighthawk)
          coffal1: 0.2050
           burowl: 0.1647
          whnjay1: 0.0471
          relser1: 0.0350
          sobcac1: 0.0280

True: gretho2 (Greater Thornbird)
          grasal3: 0.0499
           grekis: 0.0377
           bncfly: 0.0354
          palhor3: 0.0314
          yeofly1: 0.0299

True: osprey (Osprey)
           osprey: 0.9436 <-- CORRECT
           whtdov: 0.0092
           brnowl: 0.0061
          greant1: 0.0040
           greyel: 0.0033

True: greant1 (Great Antshrike)
          greant1: 0.8290 <-- CORRECT
          barant1: 0.3169
          sobcac1: 0.0400
           bkcdon: 0.0135
          grasal3: 0.0050

Stage 1 complete!


In [10]:
import torch
import timm

model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=234).cuda()

for bs in [32, 64, 128, 256, 512]:
    try:
        torch.cuda.empty_cache()
        dummy = torch.randn(bs, 3, 128, 313).cuda()
        out = model(dummy)
        loss = out.sum()
        loss.backward()
        mem = torch.cuda.max_memory_allocated() / 1024**3
        print(f"Batch size {bs:>4d}: OK  (peak GPU memory: {mem:.1f} GB / 48 GB)")
        del dummy, out, loss
    except RuntimeError as e:
        print(f"Batch size {bs:>4d}: OUT OF MEMORY")
        break

del model
torch.cuda.empty_cache()

Batch size   32: OK  (peak GPU memory: 2.2 GB / 48 GB)
Batch size   64: OK  (peak GPU memory: 4.4 GB / 48 GB)
Batch size  128: OK  (peak GPU memory: 8.6 GB / 48 GB)
Batch size  256: OK  (peak GPU memory: 17.2 GB / 48 GB)
Batch size  512: OK  (peak GPU memory: 34.3 GB / 48 GB)
